In [2]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cupy as cp
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import time

tile_size = 256
workers = 4

red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"
output_path = "ndvi_output.tif"

def compute_ndvi_gpu(red, nir):
    red_gpu = cp.asarray(red)
    nir_gpu = cp.asarray(nir)
    
    ndvi_gpu = (nir_gpu - red_gpu) / (nir_gpu + red_gpu + 1e-6)
    
    return cp.asnumpy(ndvi_gpu).astype("float32")

def process_tile(task, red_src, nir_src):
    row, col, width, height = task
    window = Window(col, row, width, height)
    
    red = red_src.read(1, window=window, boundless=True, fill_value=0).astype("float32")
    nir = nir_src.read(1, window=window, boundless=True, fill_value=0).astype("float32")
    
    ndvi = compute_ndvi_gpu(red, nir)
    return row, col, ndvi

def main():
    start = time.time()

    with rasterio.open(red_path) as src_ref:
        profile = src_ref.profile
        h, w = src_ref.height, src_ref.width
        
        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            compress="LZW",
            tiled=True,
            blockxsize=256,
            blockysize=256,
        )

        tasks = []
        for row in range(0, h, tile_size):
            for col in range(0, w, tile_size):
                win_w = min(tile_size, w - col)
                win_h = min(tile_size, h - row)
                tasks.append((row, col, win_w, win_h))

    with rasterio.open(red_path) as red_src, \
         rasterio.open(nir_path) as nir_src, \
         rasterio.open(output_path, "w", **profile) as dst:

        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = [executor.submit(process_tile, task, red_src, nir_src) for task in tasks]
            
            for future in tqdm(futures, total=len(tasks)):
                row, col, ndvi = future.result()
                dst.write(ndvi, 1, window=Window(col, row, ndvi.shape[1], ndvi.shape[0]))

    print(f"Processing finished in {time.time() - start:.2f} seconds")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'rasterio'